<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/beginner/01_python_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Exercises for Data Scientists

This notebook covers Python patterns essential for professional DS work. Run each cell, understand the output, and do the exercises.

**Topics:** Comprehensions, generators, decorators, OOP, functional programming

## 1. Comprehensions

In [ ]:
# List comprehension: [expression for item in iterable if condition]
squares = [x**2 for x in range(10)]
evens   = [x for x in range(20) if x % 2 == 0]
print('Squares:', squares)
print('Evens:', evens)

# Nested comprehension: flatten a 2D list
matrix = [[1,2,3],[4,5,6],[7,8,9]]
flat = [val for row in matrix for val in row]
print('Flattened:', flat)

# Dict comprehension
word_lengths = {word: len(word) for word in ['data', 'science', 'ML', 'AI']}
print('Word lengths:', word_lengths)

# Set comprehension (unique values)
data = [1,2,2,3,3,3,4]
unique_squares = {x**2 for x in data}
print('Unique squares:', unique_squares)

# Conditional expression (ternary)
values = [-3, -1, 0, 1, 5]
labels = ['negative' if x < 0 else 'zero' if x == 0 else 'positive' for x in values]
print('Labels:', labels)

## 2. Generators — Memory-Efficient Iteration

In [ ]:
import sys

# List vs generator memory comparison
n = 1_000_000
list_comp = [x**2 for x in range(n)]
gen_expr  = (x**2 for x in range(n))  # parentheses, not brackets

print(f'List memory: {sys.getsizeof(list_comp):,} bytes')
print(f'Generator memory: {sys.getsizeof(gen_expr)} bytes  ← constant regardless of n!')

# Generator function with yield
def data_batches(data, batch_size=3):
    """Yield data in batches — essential for large ML datasets."""
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

dataset = list(range(10))
for batch in data_batches(dataset, batch_size=3):
    print('Batch:', batch)

# Infinite generator
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print('First 10 Fibonacci:', first_10)

## 3. Decorators

In [ ]:
import time
import functools

# Decorator: a function that wraps another function
def timer(func):
    """Measure execution time of any function."""
    @functools.wraps(func)  # preserves function metadata
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'{func.__name__} took {elapsed:.4f}s')
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(10_000_000)
print('Sum:', result)

# Memoization decorator
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache  # expose cache for inspection
    return wrapper

@timer
@memoize
def fib_recursive(n):
    if n <= 1: return n
    return fib_recursive(n-1) + fib_recursive(n-2)

print('fib(35):', fib_recursive(35))  # fast because of memoization

# Python's built-in cache decorator
from functools import lru_cache

@lru_cache(maxsize=128)
def fib_lru(n):
    if n <= 1: return n
    return fib_lru(n-1) + fib_lru(n-2)

print('fib_lru(50):', fib_lru(50))

## 4. Object-Oriented Programming for Data Science

In [ ]:
import numpy as np

class DataScaler:
    """A custom StandardScaler — demonstrates OOP in DS."""
    
    def __init__(self, with_mean=True, with_std=True):
        self.with_mean = with_mean
        self.with_std = with_std
        self.mean_ = None
        self.std_ = None
        self._is_fitted = False
    
    def fit(self, X):
        X = np.array(X)
        self.mean_ = X.mean(axis=0) if self.with_mean else np.zeros(X.shape[1])
        self.std_ = X.std(axis=0) if self.with_std else np.ones(X.shape[1])
        self._is_fitted = True
        return self  # enables chaining: scaler.fit(X).transform(X)
    
    def transform(self, X):
        if not self._is_fitted:
            raise RuntimeError('Call fit() before transform()')
        X = np.array(X, dtype=float)
        return (X - self.mean_) / (self.std_ + 1e-8)
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
    def inverse_transform(self, X_scaled):
        return X_scaled * self.std_ + self.mean_
    
    def __repr__(self):
        return f'DataScaler(with_mean={self.with_mean}, with_std={self.with_std})'
    
    def __str__(self):
        if self._is_fitted:
            return f'DataScaler (fitted: mean={self.mean_.round(2)}, std={self.std_.round(2)})'
        return 'DataScaler (not fitted)'

# Usage
np.random.seed(42)
X_train = np.random.normal([10, 100], [2, 15], (100, 2))
X_test  = np.random.normal([10, 100], [2, 15], (20, 2))

scaler = DataScaler()
print(scaler)  # not fitted

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # use TRAIN statistics!

print(scaler)  # fitted
print(f'\nTrain: mean={X_train_scaled.mean(axis=0).round(3)}, std={X_train_scaled.std(axis=0).round(3)}')
print(f'Test:  mean={X_test_scaled.mean(axis=0).round(3)}, std={X_test_scaled.std(axis=0).round(3)}')

## 5. Functional Programming Tools

In [ ]:
from functools import reduce

data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map: apply function to every element
squared = list(map(lambda x: x**2, data))
print('map (squared):', squared)

# filter: keep elements matching condition
evens = list(filter(lambda x: x % 2 == 0, data))
print('filter (evens):', evens)

# reduce: fold list to single value
product = reduce(lambda a, b: a * b, data)
print('reduce (product):', product)

# Prefer list comprehensions over map/filter for readability:
squared_comp = [x**2 for x in data]          # clearer than map
evens_comp   = [x for x in data if x % 2 == 0]  # clearer than filter

# zip: pair elements from multiple iterables
names  = ['Alice', 'Bob', 'Carol']
scores = [92, 85, 95]
grades = ['A', 'B', 'A+']

for name, score, grade in zip(names, scores, grades):
    print(f'  {name}: {score} ({grade})')

# Unpacking with *
first, *middle, last = [10, 20, 30, 40, 50]
print(f'first={first}, middle={middle}, last={last}')

# enumerate
fruits = ['apple', 'banana', 'cherry']
for i, fruit in enumerate(fruits, start=1):
    print(f'{i}. {fruit}')

## 6. Context Managers

In [ ]:
from contextlib import contextmanager
import time

# Built-in context manager: with statement
# with open('file.txt', 'w') as f: f.write('...')
# File is automatically closed even if exception occurs

# Custom context manager using decorator
@contextmanager
def timer_context(label):
    start = time.perf_counter()
    try:
        yield  # execution happens here
    finally:
        elapsed = time.perf_counter() - start
        print(f'[{label}] {elapsed:.4f}s')

with timer_context('List sum'):
    result = sum(range(1_000_000))

with timer_context('NumPy sum'):
    import numpy as np
    result = np.arange(1_000_000).sum()

# Context manager class
class managed_experiment:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f'Starting experiment: {self.name}')
        self.start = time.time()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'Experiment done: {time.time() - self.start:.3f}s')
        return False  # don't suppress exceptions
    def log(self, msg):
        print(f'  [{self.name}] {msg}')

with managed_experiment('baseline_model') as exp:
    time.sleep(0.1)  # simulate work
    exp.log('Trained on 1000 samples')
    exp.log('Accuracy: 0.85')

## 7. args, kwargs, and Argument Handling

In [ ]:
# *args: variable number of positional arguments
def summary_stats(*values):
    """Compute summary stats for any number of values."""
    import numpy as np
    arr = np.array(values)
    return {'mean': arr.mean(), 'std': arr.std(), 'min': arr.min(), 'max': arr.max()}

print(summary_stats(1, 5, 3, 8, 2, 9))

# **kwargs: variable keyword arguments
def build_model(**hyperparams):
    """Build a model with arbitrary hyperparameters."""
    defaults = {'n_estimators': 100, 'max_depth': 5, 'lr': 0.1}
    params = {**defaults, **hyperparams}  # kwargs override defaults
    print('Model params:', params)
    return params

build_model(n_estimators=200, lr=0.01)

# Combining: positional, *args, keyword, **kwargs
def flexible_function(required, *args, keyword='default', **kwargs):
    print(f'required={required}')
    print(f'args={args}')
    print(f'keyword={keyword}')
    print(f'kwargs={kwargs}')

print('---')
flexible_function('hello', 1, 2, 3, keyword='custom', extra=True, n=42)

## Exercises

1. Write a list comprehension that returns all prime numbers under 100
2. Create a generator that yields Fibonacci numbers until they exceed 1000
3. Write a `retry` decorator that retries a function up to 3 times if it raises an exception
4. Implement a `Pipeline` class with `fit(X, y)`, `transform(X)`, and `fit_transform(X, y)` that chains multiple transformers
5. Write a context manager that captures and prints all print statements made inside its block